In [ ]:
!pip install pyiqa

In [ ]:
import torch
import gc
from pathlib import Path
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from sentence_transformers import SentenceTransformer, util

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def flush():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Load BLIP
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32
).to(DEVICE)

# Load SBERT
sim_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

In [ ]:

@torch.no_grad()
def get_blip_alignment(image: Image.Image, prompt: str):

    inputs = blip_processor(images=image, return_tensors="pt").to(DEVICE)

    if DEVICE.type == "cuda":
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

    out = blip_model.generate(
        **inputs,
        max_new_tokens=40,
        num_beams=5,
        repetition_penalty=2.0
    )

    caption = blip_processor.decode(out[0], skip_special_tokens=True)

    embeddings = sim_model.encode([prompt, caption], convert_to_tensor=True)
    score = util.cos_sim(embeddings[0], embeddings[1]).item()

    return caption, round(score, 4)

if __name__ == "__main__":

    IMAGE_FILE = Path("/kaggle/input/datasets/mdsunzidulislam/fuzzy-f9/seed_42_stage2_refined.png")

    try:
        image = Image.open(IMAGE_FILE).convert("RGB")
    except Exception as e:
        print(f"Failed to open {IMAGE_FILE}: {e}")
        flush()
        exit(1)

    prompt = "A dog slightly behind a chair"

    caption, score = get_blip_alignment(image, prompt)

    print("--- BLIP Evaluation Result ---")
    print("Caption:", caption)
    print("Score:", score)

    flush()

In [ ]:
import torch
from PIL import Image
from pathlib import Path
from pyiqa import create_metric
import torchvision.transforms as transforms

image_path = Path("/kaggle/input/datasets/mdsunzidulislam/fuzzy-f9/seed_42_stage2_refined.png")
if not image_path.exists():
    raise FileNotFoundError(f"⚠️ Image not found: {image_path}")

img = Image.open(image_path).convert("RGB")
print(f"✓ Image loaded: {image_path.name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models_to_test = [
    'clipiqa+',
    'clipiqa+_vitL14_512',
    'maniqa',
    'musiq',
]

iqa_metrics = {}
for model_name in models_to_test:
    try:
        iqa_metrics[model_name] = create_metric(model_name, device=device)
        print(f"✓ {model_name} loaded")
    except Exception as e:
        print(f"✗ {model_name} failed: {e}")

def get_quality_tier(score):
    if score >= 85:
        return " Exceptional"
    elif score >= 80:
        return " Excellent"
    elif score >= 75:
        return " Very Good"
    elif score >= 60:
        return " Good"
    elif score >= 40:
        return " Fair"
    else:
        return " Poor"

def evaluate_image(img):
    results = {}
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    img_tensor = transform(img).unsqueeze(0).to(device)

    for model_name, metric in iqa_metrics.items():
        try:
            score = metric(img_tensor).item() * 100
            results[model_name] = {
                "score_100": score,
                "quality_tier": get_quality_tier(score)
            }
        except:
            results[model_name] = None
    return results


In [ ]:
scores = evaluate_image(img)

print("\n CLIP-IQA Scores for the Image:")
for model_name, result in scores.items():
    if result:
        print(f"  {model_name:25s}: {result['score_100']:6.2f}/100  {result['quality_tier']}")

In [ ]:
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import gc
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def flush():
    gc.collect()
    torch.cuda.empty_cache()

# Load CLIP model (ViT-B/32)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
@torch.no_grad()
def get_clip_score(image: Image.Image, prompt: str):
    inputs = clip_processor(
        text=[prompt],
        images=image,
        return_tensors="pt",
        padding=True
    ).to(DEVICE)
    
    outputs = clip_model(**inputs)
    
    # Cosine similarity between image and text embeddings
    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds
    
    # Normalize embeddings
    image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)
    text_embeds = text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)
    
    score = (image_embeds @ text_embeds.T).item()  # Cosine similarity
    
    return round(score, 4)

In [ ]:
if __name__ == "__main__":
    # Example: replace with your generated image
    from PIL import Image
    test_prompt = "A dog slightly behind a chair"    
    image = Image.open("/kaggle/input/datasets/mdsunzidulislam/fuzzy-f9/seed_42_stage2_refined.png").convert("RGB")
    
    score = get_clip_score(image, test_prompt)
    
    print(f"--- CLIP Evaluation Result ---")
    print(f"Prompt: {test_prompt}")
    print(f"Score:  {score}")
    
    flush()